In [2]:
# After loading kaggle_images.csv

import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\Manojkumar Mohankuma\OneDrive\Desktop\InfinyAI\Computer Vision & Machine Learning for Automated Sorting of Precious-Metal Scrap"
)
NOTEBOOK_DIR = PROJECT_DIR / "notebook"
KAGGLE_CSV_PATH = NOTEBOOK_DIR / "kaggle_images.csv"

print("CSV path:", KAGGLE_CSV_PATH)
print("Exists:", KAGGLE_CSV_PATH.exists())

kaggle_df = pd.read_csv(KAGGLE_CSV_PATH)

classes = sorted(kaggle_df["class"].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
print("Classes:", classes)

def load_split_as_arrays(df, split_name, image_size=(64, 64)):
    split_df = df[df["split"] == split_name].copy()
    if split_df.empty:
        print(f"No images found for split '{split_name}'")
        return None, None

    X_list = []
    y_list = []

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Loading {split_name}"):
        img_path = Path(row["path"])
        cls_name = row["class"]
        label = class_to_idx[cls_name]

        img = Image.open(img_path).convert("RGB")
        img = img.resize(image_size)
        arr = np.array(img, dtype=np.float32) / 255.0  # normalize to [0, 1]

        X_list.append(arr)
        y_list.append(label)

    X = np.stack(X_list, axis=0)  # (N, H, W, 3)
    y = np.array(y_list, dtype=np.int64)

    print(f"{split_name}: X shape = {X.shape}, y shape = {y.shape}")
    return X, y

X_train, y_train = load_split_as_arrays(kaggle_df, "train", image_size=(64, 64))
X_val,   y_val   = load_split_as_arrays(kaggle_df, "val",   image_size=(64, 64))
X_test,  y_test  = load_split_as_arrays(kaggle_df, "test",  image_size=(64, 64))

CSV path: C:\Users\Manojkumar Mohankuma\OneDrive\Desktop\InfinyAI\Computer Vision & Machine Learning for Automated Sorting of Precious-Metal Scrap\notebook\kaggle_images.csv
Exists: True
Classes: ['battery', 'keyboard', 'microwave', 'mobile', 'mouse', 'pcb', 'player', 'printer', 'television', 'washing_machine']


Loading train: 100%|██████████| 2400/2400 [00:29<00:00, 80.18it/s] 


train: X shape = (2400, 64, 64, 3), y shape = (2400,)


Loading val: 100%|██████████| 300/300 [00:03<00:00, 81.64it/s]


val: X shape = (300, 64, 64, 3), y shape = (300,)


Loading test: 100%|██████████| 300/300 [00:03<00:00, 81.69it/s]

test: X shape = (300, 64, 64, 3), y shape = (300,)


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from pathlib import Path

num_classes = len(classes)

def build_simple_cnn(input_shape=(64, 64, 3), num_classes=10):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation="relu", input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax"),
    ])
    return model

cnn_model = build_simple_cnn(input_shape=(64, 64, 3), num_classes=num_classes)

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_model.summary()

history = cnn_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_val, y_val),
)

c:\Users\Manojkumar Mohankuma\OneDrive\Desktop\InfinyAI\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 684,490 (2.61 MB)

 Trainable params: 684,490 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.1787 - loss: 2.2198 - val_accuracy: 0.3467 - val_loss: 1.9228
Epoch 2/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.3354 - loss: 1.8869 - val_accuracy: 0.5033 - val_loss: 1.5363
Epoch 3/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4271 - loss: 1.6463 - val_accuracy: 0.5567 - val_loss: 1.4383
Epoch 4/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.4771 - loss: 1.4511 - val_accuracy: 0.6267 - val_loss: 1.2752
Epoch 5/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5550 - loss: 1.3090 - val_accuracy: 0.5500 - val_loss: 1.3387
Epoch 6/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5846 - loss: 1.2341 - val_accuracy: 0.6267 - val_loss: 1.1564
Epoch 7/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6179 - loss: 1.1092 - val_accuracy: 0.6800 - val_loss: 1.0587
Epoch 8/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.6608 - loss: 0.9886 - val_accuracy: 0.6867 - v

In [5]:
from sklearn.metrics import classification_report, confusion_matrix

# Evaluate on test set
test_loss, test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy (CNN):", test_acc)

y_pred_prob = cnn_model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)

print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

# Save model
models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "cnn_e_waste.h5"
cnn_model.save(model_path)
print("Saved CNN model to:", model_path)

Test accuracy (CNN): 0.6266666650772095
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


                 precision    recall  f1-score   support

        battery       0.43      0.67      0.53        30
       keyboard       0.95      0.70      0.81        30
      microwave       0.67      0.73      0.70        30
         mobile       0.70      0.53      0.60        30
          mouse       0.48      0.50      0.49        30
            pcb       0.74      0.67      0.70        30
         player       0.53      0.57      0.55        30
        printer       0.62      0.53      0.57        30
     television       0.62      0.67      0.65        30
washing_machine       0.75      0.70      0.72        30

       accuracy                           0.63       300
      macro avg       0.65      0.63      0.63       300
   weighted avg       0.65      0.63      0.63       300

Confusion matrix:
 [[20  0  2  0  2  2  1  1  0  2]
 [ 5 21  0  0  1  0  2  1  0  0]
 [ 1  0 22  0  1  0  2  1  2  1]
 [ 1  1  3 16  2  0  3  0  4  0]
 [ 5  0  0  4 15  0  2  2  1  1]
 [ 6  0  0  2  